# NB14 — Error Analysis

Confusion matrix, length-stratified accuracy, proposed-vs-author-DL disagreement, and qualitative error
samples for the proposed model on **Ben-Sarc binary**. CPU/Mac.


In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
def find_repo_root():
    for c in [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents):
        if (c/"01_data").exists(): return c.resolve()
    raise RuntimeError("repo root not found.")
ROOT=find_repo_root(); TABLES=ROOT/"04_outputs"/"tables"; PRED=ROOT/"04_outputs"/"predictions"; FIGS=ROOT/"04_outputs"/"figures"
for p in [TABLES,FIGS]: p.mkdir(parents=True,exist_ok=True)
TASK="ben_sarc_binary"
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
def collect(task, split):
    out={}
    for f in sorted(PRED.glob(f"*_{task}_{split}_predictions.csv")):
        name=f.name.replace(f"_{task}_{split}_predictions.csv","")
        try: out[name]=pd.read_csv(f).dropna(subset=["text"]).drop_duplicates("text")
        except Exception as e: print("skip",f.name,e)
    return out
def macro(df): return f1_score(df["gold_label"],df["pred_label"],average="macro",zero_division=0)
print("ROOT:",ROOT)

ROOT: /Users/sefayet/Desktop/Github/Sarcasm_detection


In [2]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
test=collect(TASK,"test")
PROPOSED=next((c for c in ["09b_fgm_awp","09_FINAL_proposed","08_P1_fullft_fgm","07_fgm_e05"] if c in test),None)
assert PROPOSED, "no proposed predictions"
dp=test[PROPOSED].copy(); y=dp["gold_label"].values; pr=dp["pred_label"].values
print("proposed:",PROPOSED,"| macro-F1:",round(macro(dp),4))
cmx=confusion_matrix(y,pr); print("confusion:\n",cmx)
json.dump({"labels":[0,1],"matrix":cmx.tolist()},open(TABLES/"14_confusion.json","w"),indent=2)

# length-stratified
dp["len"]=dp["text"].astype(str).str.len()
dp["correct_"]=(y==pr).astype(int)
bins=[0,40,80,120,200,10000]; lab=["0-40","40-80","80-120","120-200","200+"]
dp["bucket"]=pd.cut(dp["len"],bins=bins,labels=lab)
ls=dp.groupby("bucket").agg(n=("correct_","size"),acc=("correct_","mean")).reset_index()
ls.to_csv(TABLES/"14_length_stratified.csv",index=False)
print("\nlength-stratified accuracy:\n",ls.to_string(index=False))

proposed: 09b_fgm_awp | macro-F1: 0.8038
confusion:
 [[1081  200]
 [ 302  980]]

length-stratified accuracy:
  bucket    n      acc
   0-40  649 0.787365
  40-80 1048 0.794847
 80-120  505 0.821782
120-200  274 0.854015
   200+   87 0.781609


In [3]:
# disagreement vs author DL (if present)
dl=next((k for k in test if k.startswith("03_")),None)
if dl:
    da=test[dl].set_index("text"); db=dp.set_index("text"); idx=sorted(set(da.index)&set(db.index))
    yv=db.loc[idx,"gold_label"].values
    cp=(db.loc[idx,"pred_label"].values==yv); cd=(da.loc[idx,"pred_label"].values==yv)
    tab=pd.DataFrame({"both_correct":[int((cp&cd).sum())],"only_proposed":[int((cp&~cd).sum())],
                      "only_authorDL":[int((~cp&cd).sum())],"both_wrong":[int((~cp&~cd).sum())]})
    tab.to_csv(TABLES/"14_disagreement.csv",index=False)
    print("\nproposed vs author-DL ("+dl+"):\n",tab.to_string(index=False))
else:
    print("\n(no 03_* author-DL predictions found -> disagreement table skipped)")

# qualitative: high-confidence errors
pcols=[c for c in dp.columns if c.startswith("prob_")]
if pcols:
    dp["conf"]=dp[pcols].max(1)
    err=dp[dp["correct_"]==0].sort_values("conf",ascending=False).head(15)[["text","gold_label","pred_label","conf"]]
    err.to_csv(TABLES/"14_high_conf_errors.csv",index=False)
    print("\ntop high-confidence errors saved -> 14_high_conf_errors.csv")

plt.figure(figsize=(5,4)); plt.bar(ls["bucket"].astype(str),ls["acc"])
plt.ylim(0,1); plt.ylabel("accuracy"); plt.xlabel("comment length (chars)")
plt.title("Accuracy by comment length"); plt.tight_layout()
plt.savefig(FIGS/"14_length_accuracy.png",dpi=160); plt.close()
print("saved -> 04_outputs/figures/14_length_accuracy.png")


(no 03_* author-DL predictions found -> disagreement table skipped)

top high-confidence errors saved -> 14_high_conf_errors.csv
saved -> 04_outputs/figures/14_length_accuracy.png
